In [1]:
import sys, os
import xarray as xr
import numpy as np

In [2]:
sys.path

['/home/563/fm6730/home',
 '/home/563/fm6730/ondemand/data/sys/dashboard/batch_connect/sys/jupyter/ncigadi/output/7aaae13e-38b5-43d0-bf0c-dbe60bf4b3b7/lib/python3',
 '/g/data/xp65/public/apps/med_conda/envs/analysis3-26.05/lib/python312.zip',
 '/g/data/xp65/public/apps/med_conda/envs/analysis3-26.05/lib/python3.12',
 '/g/data/xp65/public/apps/med_conda/envs/analysis3-26.05/lib/python3.12/lib-dynload',
 '',
 '/g/data/xp65/public/apps/med_conda/envs/analysis3-26.05/lib/python3.12/site-packages']

In [3]:
from pathlib import Path

solar_cf = Path("/home/563/fm6730/localrepo/GC26-combined-solar-wind/data/raw/solar_cf/")

# List files
print(list(solar_cf.iterdir()))

# Example file
# file = solar_cf / "my_file.nc"

# print(file.exists())

[PosixPath('/home/563/fm6730/localrepo/GC26-combined-solar-wind/data/raw/solar_cf/solar_capacity_factor_van_der_Wiel_era5_hourly_1980_Aus.nc'), PosixPath('/home/563/fm6730/localrepo/GC26-combined-solar-wind/data/raw/solar_cf/solar_capacity_factor_van_der_Wiel_era5_hourly_2001_Aus.nc'), PosixPath('/home/563/fm6730/localrepo/GC26-combined-solar-wind/data/raw/solar_cf/solar_capacity_factor_van_der_Wiel_era5_hourly_1949_Aus.nc'), PosixPath('/home/563/fm6730/localrepo/GC26-combined-solar-wind/data/raw/solar_cf/solar_capacity_factor_van_der_Wiel_era5_hourly_1981_Aus.nc'), PosixPath('/home/563/fm6730/localrepo/GC26-combined-solar-wind/data/raw/solar_cf/solar_capacity_factor_van_der_Wiel_era5_hourly_1985_Aus.nc'), PosixPath('/home/563/fm6730/localrepo/GC26-combined-solar-wind/data/raw/solar_cf/solar_capacity_factor_van_der_Wiel_era5_hourly_1989_Aus.nc'), PosixPath('/home/563/fm6730/localrepo/GC26-combined-solar-wind/data/raw/solar_cf/solar_capacity_factor_van_der_Wiel_era5_hourly_1950_Aus.nc')

In [4]:
year = 2003

In [5]:
folder_main = '/g/data/w42/dr6273/work/projects/Aus_energy/production_metrics'

In [6]:
folder_solar = f'{folder_main}/solar/capacity_factor/van_der_Wiel' 
folder_wind  = f'{folder_main}/wind/capacity_factor/van_der_Wiel' 

In [7]:
file_solar   = f'{folder_solar}/solar_capacity_factor_van_der_Wiel_era5_hourly_{year}_Aus.nc'
file_wind    = f'{folder_wind}/wind_capacity_factor_van_der_Wiel_era5_hourly_{year}_Aus.nc'

In [8]:
ds_solar = xr.open_dataset(file_solar)
ds_wind  = xr.open_dataset(file_wind)

In [9]:
da_solar = ds_solar['capacity_factor']
da_wind  = ds_wind['capacity_factor']

In [ ]:
da_solar_sel = xr.where(da_solar > 0.1, 1, 0) # 1 is higher than 0.1
da_wind_sel  = xr.where(da_wind > 0.1, 1, 0) # 1 is higher than 0.1

In [ ]:
da_combined = da_solar_sel + da_wind_sel

In [ ]:
da_combined = xr.where(da_combined == 0, 1, 0) # 1 is combined solar & wind lulls

In [ ]:
da_combined

In [ ]:
da_sum_combined = da_combined.sum(dim='time')

In [ ]:
import pandas as pd

In [ ]:
da_sum_combined_wcoords = da_sum_combined.expand_dims(year=[int(year)]).assign_coords(year=[int(year)])

In [ ]:

#Outputting

ds_out = xr.Dataset({
    'count_hour': da_sum_combined_wcoords,
    'percentage': da_sum_combined_wcoords/len(da_combined.time)*100.
    })

In [ ]:
ds_out

In [ ]:

folder_out   = Path("/home/563/fm6730/localrepo/GC26-combined-solar-wind/data/processed/")

In [ ]:

da_sum_combined.to_netcdf(f"{folder_out}/days_capacity_factor_lower_than_10pc_{year}.nc")